# Fixtures API

Collects the 2026/27 La Liga fixtures and results from the football-data.org API, saves the raw JSON and loads the matches into the `fixtures` table in PostgreSQL. The predictions for these fixtures are made later, in the modeling notebook.

Football data provided by the Football-Data.org API.

---

## Imports


In [7]:
import json
import os
import sys
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

## Notebook setup

Find the repo root, so paths work from any clone and `src/` is importable.


In [2]:
REPO_ROOT = next(
    folder
    for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "environment.yml").exists()
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

Repo root: c:\football-score-predictor


## Config / constants


In [8]:
# football-data.org API, version 4. "PD" (Primera Division) is its code for La Liga
FOOTBALL_DATA_BASE_URL = "https://api.football-data.org/v4"
COMPETITION_CODE = "PD"

# The API names a season by its start year, like our SeasonStart column (2026 = 2026/27).
# Asking for it explicitly keeps the notebook on 2026/27 even after the API moves to the next season
API_SEASON = 2026

# Stop waiting after 30 seconds, so a slow server can't freeze the notebook forever
REQUEST_TIMEOUT_SECONDS = 30

# Raw API responses are saved here. data/raw/ is git-ignored, and the API's terms don't
# allow republishing its data, so the JSON stays locally
FOOTBALL_DATA_RAW_DIR = REPO_ROOT / "data" / "raw" / "football_data"

# Local time zone for timestamps
LOCAL_TIMEZONE = ZoneInfo("Asia/Jerusalem")

## API connection


In [4]:
load_dotenv(REPO_ROOT / ".env")

FOOTBALL_DATA_API_KEY = os.environ["FOOTBALL_DATA_API"]

print(f"API key loaded: {len(FOOTBALL_DATA_API_KEY) > 0}")

API key loaded: True


## Fetch the season's matches

One request returns every match of the season: Finished ones with their score, upcoming ones with their kickoff date. The free plan allows 10 requests per minute, and this notebook makes one per run.


In [ ]:
def fetch_competition_matches(competition_code: str, season: int, api_key: str) -> dict:
    """
    Fetches all matches of one competition season from the football-data.org API.

    Args:
        competition_code (str): The API's competition code, e.g "PD" for La Liga.
        season (int): The season's start year, e.g 2026 for 2026/27.
        api_key (str): The football-data.org API key, sent in the X-Auth-Token header.

    Returns:
        dict: The parsed JSON response. Its "matches" key hold one dict per match,
            with the kickoff time, status, teams, and score.
    """
    url = f"{FOOTBALL_DATA_BASE_URL}/competitions/{competition_code}/matches"
    response = requests.get(
        url,
        headers={"X-Auth-Token": api_key},
        params={"season": season},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )

    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        raise RuntimeError(
            f"football-data.org returned {response.status_code}: {response.text}"
        ) from error

    print(
        f"Requests left this minute: {response.headers.get('X-Requests-Available-Minute')}"
    )

    return response.json()

In [6]:
matches_response = fetch_competition_matches(
    COMPETITION_CODE, API_SEASON, FOOTBALL_DATA_API_KEY
)

print(matches_response["filters"])
print(matches_response["resultSet"])

Requests left this minute: 9
{'season': 2026}
{'count': 380, 'first': '2026-08-15', 'last': '2027-05-30', 'played': 119}


In [ ]:
# One file per run, named with the UTC time of the call. Older files are kept for history of snapshots.
fetched_at = datetime.now(LOCAL_TIMEZONE)
FOOTBALL_DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
raw_path = (
    FOOTBALL_DATA_RAW_DIR
    / f"{COMPETITION_CODE}_matches_{API_SEASON}_{fetched_at:%Y%m%d%H%M%S%z}.json"
)

raw_path.write_text(
    json.dumps(matches_response, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Saved {raw_path.name} ({raw_path.stat().st_size / 1024:.0f} KB)")

Saved PD_matches_2026_20261117152440+0200.json (594 KB)


In [11]:
api_matches = pd.json_normalize(matches_response["matches"])

print(api_matches.shape)

api_matches["status"].value_counts()

(380, 39)


status
SCHEDULED    228
FINISHED     119
TIMED         32
CANCELLED      1
Name: count, dtype: int64